# Engenharia de SPECS Analíticas com Apache Spark (PySpark)
**Tech Challenge - Fase 3 | Pipeline de Engenharia de Dados**

---

### 🎯 Objetivo
Este notebook realiza a ingestão da **SOT (Source of Truth)** oficial do *State of Data Brasil* (`dados/base_consolidada.parquet`), executa o pipeline de **engenharia de features, normalização salarial e categorização dimensional com PySpark**, e persiste as **7 SPECS Analíticas** otimizadas em formato Parquet na pasta `dados/bases_analiticas/`.

```text
dados/base_consolidada.parquet (SOT Auditável - 14.005 respondentes)
       │
       ▼ [Apache Spark (PySpark) Engine]
       ├── 1. spec_respondentes.parquet                  -> Perfil demográfico completo, macro-cargos e salários
       ├── 2. spec_adocao_tecnologias.parquet            -> Stack tecnológico detalhado (Linguagens, Cloud, Bancos, ETL)
       ├── 3. spec_adocao_ia.parquet                     -> Adoção de IA (Uso Individual vs Corporativo vs Barreiras)
       ├── 4. spec_diversidade_carreira.parquet          -> Métricas agregadas de equidade, efeito funil e pay gap
       ├── 5. spec_segmentacao_negocio.parquet           -> Clusters executivos para planejamento de remuneração e IA
       ├── 6. spec_dinamica_trabalho_satisfacao.parquet  -> Modelo de trabalho atual vs ideal e retenção de talentos
       └── 7. spec_estrutura_times_empresa.parquet       -> Composição e papéis das equipes de dados nas empresas
```


In [1]:
import os
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 1. Configuração automática do JAVA_HOME para execução do PySpark
if 'JAVA_HOME' not in os.environ:
    java_candidate = r'C:\Program Files\Eclipse Adoptium\jdk-17.0.20.101-hotspot'
    if os.path.exists(java_candidate):
        os.environ['JAVA_HOME'] = java_candidate
        os.environ['PATH'] = os.path.join(java_candidate, 'bin') + os.pathsep + os.environ.get('PATH', '')

import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
import pandas as pd

# 2. Configuração de caminhos
BASE_PROJETO = Path.cwd()
ORIGEM_SOT = BASE_PROJETO / 'dados' / 'base_consolidada.parquet'
SAIDA_SPECS = BASE_PROJETO / 'dados' / 'bases_analiticas'

if not ORIGEM_SOT.exists():
    ORIGEM_SOT = BASE_PROJETO / 'projeto' / 'fase_3_data_analytics' / 'dados' / 'base_consolidada.parquet'
    SAIDA_SPECS = BASE_PROJETO / 'projeto' / 'fase_3_data_analytics' / 'dados' / 'bases_analiticas'

if not ORIGEM_SOT.exists():
    ORIGEM_SOT = Path('D:/d/pos/Fase 3/projeto/fase_3_data_analytics/dados/base_consolidada.parquet')
    SAIDA_SPECS = Path('D:/d/pos/Fase 3/projeto/fase_3_data_analytics/dados/bases_analiticas')

assert ORIGEM_SOT.exists(), f'SOT não encontrada: {ORIGEM_SOT}'
SAIDA_SPECS.mkdir(parents=True, exist_ok=True)

# 3. Inicialização da SparkSession
print("Inicializando SparkSession...")
spark = SparkSession.builder \
    .appName("Engenharia_SPECS_PySpark") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.ui.showConsoleProgress", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# 4. Leitura da SOT via PySpark
print(f"Lendo SOT a partir de: {ORIGEM_SOT}")
df_spark = spark.read.parquet(str(ORIGEM_SOT))
print(f"[OK] Registros carregados: {df_spark.count():,} | Colunas: {len(df_spark.columns)}")


Inicializando SparkSession...
Lendo SOT a partir de: d:\d\pos\Fase 3\projeto\fase_3_data_analytics\dados\base_consolidada.parquet
[OK] Registros carregados: 14,005 | Colunas: 135


In [3]:
# 5. Transformação de Features, Salário e Normalização Dimensional
df_spark = df_spark.withColumn("id_str", F.col("id").cast(T.StringType())) \
                   .withColumn("ano_pesquisa", F.col("ano_pesquisa").cast(T.IntegerType())) \
                   .withColumn("respondente_key", F.concat_ws("_", F.col("ano_pesquisa"), F.col("id_str")))

# Normalização Salarial Numérica
df_spark = df_spark.withColumn(
    "salario_estimado_num",
    F.when(F.col("faixa_salarial") == "Menos de R$ 1.000/mês", 1000.0)
     .when(F.col("faixa_salarial").isin("de R$ 101/mês a R$ 2.000/mês", "de R$ 1.001/mês a R$ 2.000/mês"), 1500.0)
     .when(F.col("faixa_salarial") == "de R$ 2.001/mês a R$ 3.000/mês", 2500.0)
     .when(F.col("faixa_salarial") == "de R$ 3.001/mês a R$ 4.000/mês", 3500.0)
     .when(F.col("faixa_salarial") == "de R$ 4.001/mês a R$ 6.000/mês", 5000.0)
     .when(F.col("faixa_salarial") == "de R$ 6.001/mês a R$ 8.000/mês", 7000.0)
     .when(F.col("faixa_salarial") == "de R$ 8.001/mês a R$ 12.000/mês", 10000.0)
     .when(F.col("faixa_salarial") == "de R$ 12.001/mês a R$ 16.000/mês", 14000.0)
     .when(F.col("faixa_salarial") == "de R$ 16.001/mês a R$ 20.000/mês", 18000.0)
     .when(F.col("faixa_salarial") == "de R$ 20.001/mês a R$ 25.000/mês", 22500.0)
     .when(F.col("faixa_salarial").isin("de R$ 25.001/mês a R$ 3000/mês", "de R$ 25.001/mês a R$ 30.000/mês"), 27500.0)
     .when(F.col("faixa_salarial") == "de R$ 30.001/mês a R$ 40.000/mês", 35000.0)
     .when(F.col("faixa_salarial") == "Acima de R$ 40.001/mês", 45000.0)
     .otherwise(F.lit(None).cast(T.DoubleType()))
).withColumn(
    "ordem_salarial",
    F.when(F.col("faixa_salarial") == "Menos de R$ 1.000/mês", 1)
     .when(F.col("faixa_salarial").isin("de R$ 101/mês a R$ 2.000/mês", "de R$ 1.001/mês a R$ 2.000/mês"), 2)
     .when(F.col("faixa_salarial") == "de R$ 2.001/mês a R$ 3.000/mês", 3)
     .when(F.col("faixa_salarial") == "de R$ 3.001/mês a R$ 4.000/mês", 4)
     .when(F.col("faixa_salarial") == "de R$ 4.001/mês a R$ 6.000/mês", 5)
     .when(F.col("faixa_salarial") == "de R$ 6.001/mês a R$ 8.000/mês", 6)
     .when(F.col("faixa_salarial") == "de R$ 8.001/mês a R$ 12.000/mês", 7)
     .when(F.col("faixa_salarial") == "de R$ 12.001/mês a R$ 16.000/mês", 8)
     .when(F.col("faixa_salarial") == "de R$ 16.001/mês a R$ 20.000/mês", 9)
     .when(F.col("faixa_salarial") == "de R$ 20.001/mês a R$ 25.000/mês", 10)
     .when(F.col("faixa_salarial").isin("de R$ 25.001/mês a R$ 3000/mês", "de R$ 25.001/mês a R$ 30.000/mês"), 11)
     .when(F.col("faixa_salarial") == "de R$ 30.001/mês a R$ 40.000/mês", 12)
     .when(F.col("faixa_salarial") == "Acima de R$ 40.001/mês", 13)
     .otherwise(F.lit(None).cast(T.IntegerType()))
)

# Macro-Cargos
df_spark = df_spark.withColumn(
    "macro_cargo",
    F.when(F.col("cargo_atual").isNull(), "Não informado / Em transição")
     .when(F.col("cargo_atual").rlike("(?i)Engenheiro de Dados|Arquiteto de Dados|Analytics Engineer"), "Engenharia & Arquitetura de Dados")
     .when(F.col("cargo_atual").rlike("(?i)Cientista de Dados|Estatístico|Economista"), "Ciência de Dados")
     .when(F.col("cargo_atual").rlike("(?i)Analista de Dados|Analista de BI|Inteligência de Mercado"), "Análise de Dados & BI")
     .when(F.col("cargo_atual").rlike("(?i)Machine Learning|ML Engineer|AI Engineer"), "Machine Learning & IA")
     .when(F.col("cargo_atual").rlike("(?i)Analista de Negócios|Business Analyst|Product Manager|PM/APM/DPM"), "Negócios & Gestão de Produto")
     .when(F.col("cargo_atual").rlike("(?i)DBA|Administrador de Banco"), "DBA & Infraestrutura")
     .when(F.col("cargo_atual").rlike("(?i)Desenvolvedor|Engenheiro de Software|Analista de Sistemas|Outras Engenharias|Suporte"), "Engenharia de Software / Outras Engenharias")
     .when(F.col("cargo_atual").rlike("(?i)Professor|Pesquisador"), "Academia & Pesquisa")
     .otherwise("Outros")
)

# Modelos de Trabalho
df_spark = df_spark.withColumn(
    "modelo_trabalho_resumido",
    F.when(F.col("modelo_trabalho_atual").isNull(), "Não informado")
     .when(F.col("modelo_trabalho_atual").rlike("(?i)100% remoto|remoto"), "Remoto")
     .when(F.col("modelo_trabalho_atual").rlike("(?i)flexível|flexivel"), "Híbrido Flexível")
     .when(F.col("modelo_trabalho_atual").rlike("(?i)fixo|dias fixos"), "Híbrido Fixo")
     .when(F.col("modelo_trabalho_atual").rlike("(?i)presencial"), "Presencial")
     .otherwise(F.col("modelo_trabalho_atual"))
).withColumn(
    "modelo_ideal_resumido",
    F.when(F.col("modelo_trabalho_ideal").isNull(), "Não informado")
     .when(F.col("modelo_trabalho_ideal").rlike("(?i)100% remoto|remoto"), "Remoto")
     .when(F.col("modelo_trabalho_ideal").rlike("(?i)flexível|flexivel"), "Híbrido Flexível")
     .when(F.col("modelo_trabalho_ideal").rlike("(?i)fixo|dias fixos"), "Híbrido Fixo")
     .when(F.col("modelo_trabalho_ideal").rlike("(?i)presencial"), "Presencial")
     .otherwise(F.col("modelo_trabalho_ideal"))
)
print("[OK] Features e normalizacoes calculadas com sucesso!")


[OK] Features e normalizacoes calculadas com sucesso!


In [4]:
# 6. Geração e Persistência das 7 SPECS Analíticas

print("Gerando as 7 SPECS analiticas...")

# 1. spec_respondentes
dimensoes_resp = [
    'respondente_key', 'id', 'ano_pesquisa', 'genero', 'cor_raca_etnia', 'faixa_idade',
    'estado_onde_mora', 'uf_onde_mora', 'regiao_onde_mora', 'nivel_de_ensino',
    'area_de_formacao', 'setor', 'cargo_atual', 'macro_cargo', 'nivel',
    'faixa_salarial', 'salario_estimado_num', 'ordem_salarial',
    'tempo_experiencia_dados', 'oportunidade_buscada',
    'modelo_trabalho_atual', 'modelo_trabalho_resumido', 'modelo_trabalho_ideal', 'modelo_ideal_resumido'
]
cols_existentes_resp = [c for c in dimensoes_resp if c in df_spark.columns]
spark_spec_respondentes = df_spark.select(cols_existentes_resp).dropDuplicates(['respondente_key'])
spark_spec_respondentes.toPandas().to_parquet(SAIDA_SPECS / 'spec_respondentes.parquet', index=False)
print(f"  [OK] 1/7 spec_respondentes: {spark_spec_respondentes.count():,} linhas")

# 2. spec_adocao_tecnologias
familias_tecnologia = {
    'linguagem': ['sql', 'r', 'python', 'c_cpp_csharp', 'dotnet', 'java', 'julia', 'sas_stata', 'visual_basic_vba', 'scala', 'matlab', 'rust', 'php', 'javascript'],
    'banco_ou_plataforma': ['mysql', 'oracle', 'sql_server', 'amazon_aurora_rds', 'dynamodb', 'coachdb', 'cassandra', 'mongodb', 'mariadb', 'datomic', 's3', 'postgresql', 'elasticsearch', 'db2', 'microsoft_access', 'sqlite', 'sybase', 'firebase', 'vertica', 'redis', 'neo4j', 'google_bigquery', 'google_firestore', 'amazon_redshift', 'amazon_athena', 'snowflake', 'databricks', 'hbase', 'presto', 'splunk', 'sap_hana', 'hive', 'firebird'],
    'cloud': ['aws_cloud', 'gcp_cloud', 'azure_cloud', 'oracle_cloud', 'ibm', 'cloud_propria'],
    'etl': ['scripts_python', 'sql_stored_procedures', 'apache_airflow', 'apache_nifi', 'luigi', 'aws_glue', 'talend', 'pentaho', 'alteryx', 'stitch', 'fivetran', 'google_dataflow', 'oracle_data_integrator', 'ibm_datastage', 'sap_bw_etl', 'sql_server_integration_services_ssis', 'sas_data_integration', 'qlik_sense', 'knime', 'databricks_etl']
}
nomes_formatados = {
    'sql': 'SQL', 'r': 'R', 'python': 'Python', 'c_cpp_csharp': 'C/C++/C#', 'dotnet': '.NET', 'java': 'Java', 'julia': 'Julia', 'sas_stata': 'SAS/Stata', 'visual_basic_vba': 'VBA', 'scala': 'Scala', 'matlab': 'Matlab', 'rust': 'Rust', 'php': 'PHP', 'javascript': 'JavaScript', 'aws_cloud': 'AWS', 'gcp_cloud': 'Google Cloud (GCP)', 'azure_cloud': 'Azure', 'oracle_cloud': 'Oracle Cloud', 'ibm': 'IBM Cloud', 'cloud_propria': 'Cloud Própria / On-Premise', 'google_bigquery': 'Google BigQuery', 'amazon_redshift': 'Amazon Redshift', 'amazon_athena': 'Amazon Athena', 'amazon_aurora_rds': 'Amazon Aurora/RDS', 'sql_server': 'SQL Server', 'postgresql': 'PostgreSQL', 'mysql': 'MySQL', 'sqlite': 'SQLite', 'mongodb': 'MongoDB', 'snowflake': 'Snowflake', 'databricks': 'Databricks', 'scripts_python': 'Scripts Python', 'sql_stored_procedures': 'Stored Procedures SQL', 'apache_airflow': 'Apache Airflow', 'apache_nifi': 'Apache NiFi', 'aws_glue': 'AWS Glue', 'databricks_etl': 'Databricks (ETL/Workflows)', 'google_dataflow': 'Google Dataflow', 'sql_server_integration_services_ssis': 'SSIS', 'coachdb': 'CouchDB'
}
cols_contexto_tech = [c for c in ['respondente_key', 'id', 'ano_pesquisa', 'genero', 'regiao_onde_mora', 'cargo_atual', 'macro_cargo', 'nivel', 'tempo_experiencia_dados', 'faixa_salarial', 'salario_estimado_num', 'modelo_trabalho_resumido', 'setor'] if c in df_spark.columns]
tech_dfs = []
for familia, cols in familias_tecnologia.items():
    for col in cols:
        if col in df_spark.columns:
            nome_tech = nomes_formatados.get(col, col.replace('_cloud', '').replace('_etl', '').replace('_', ' ').strip().title())
            sub = df_spark.filter(F.col(col).isin("1", "1.0", "true", "True", "SIM", "Sim", "sim", "x", "X")) \
                          .select(cols_contexto_tech) \
                          .withColumn("familia", F.lit(familia)) \
                          .withColumn("tecnologia_id", F.lit(col)) \
                          .withColumn("tecnologia", F.lit(nome_tech)) \
                          .withColumn("tecnologia_principal", F.lit(False))
            tech_dfs.append(sub)

spark_spec_tech = tech_dfs[0]
for tdf in tech_dfs[1:]:
    spark_spec_tech = spark_spec_tech.unionByName(tdf)
spark_spec_tech = spark_spec_tech.dropDuplicates(['respondente_key', 'familia', 'tecnologia'])

if 'linguagem_principal' in df_spark.columns:
    df_ling = df_spark.select('respondente_key', 'linguagem_principal').filter(F.col('linguagem_principal').isNotNull()) \
                      .withColumn('ling_lower', F.lower(F.trim(F.col('linguagem_principal'))))
    spark_spec_tech = spark_spec_tech.join(df_ling, on='respondente_key', how='left') \
                                     .withColumn('tecnologia_principal', (F.col('familia') == 'linguagem') & (F.lower(F.col('tecnologia')) == F.col('ling_lower'))) \
                                     .drop('ling_lower', 'linguagem_principal')
spark_spec_tech.toPandas().to_parquet(SAIDA_SPECS / 'spec_adocao_tecnologias.parquet', index=False)
print(f"  [OK] 2/7 spec_adocao_tecnologias: {spark_spec_tech.count():,} linhas")

# 3. spec_adocao_ia
dicionario_ia = {
    'ia_solucoes_gratuitas': ('Uso Individual / Produtividade', 'Uso de Soluções Gratuitas (ChatGPT, Gemini, etc.)', 'Uso Ativo'),
    'ia_paga_usuario': ('Uso Individual / Produtividade', 'Assinatura Paga pelo Próprio Profissional', 'Uso Ativo'),
    'ia_paga_empresa': ('Uso Individual / Produtividade', 'Assinatura Paga pela Empresa', 'Uso Ativo'),
    'usa_copilot': ('Uso Individual / Produtividade', 'Uso de Assistente de Código (Copilot, Cursor, etc.)', 'Uso Ativo'),
    'ia_independente_descentralizada': ('Uso Individual / Produtividade', 'Uso Descentralizado / Autônomo por Colaboradores', 'Uso Ativo'),
    'ia_direcionamento_centralizado': ('Uso Organizacional / Negócio', 'Direcionamento Centralizado Corporativo', 'Uso Ativo'),
    'desenvolvedores_copilot': ('Uso Organizacional / Negócio', 'Equipes de Dev/Dados com Ferramentas de IA', 'Uso Ativo'),
    'ia_produtos_internos': ('Uso Organizacional / Negócio', 'IA Aplicada em Eficiência e Processos Internos', 'Uso Ativo'),
    'ia_produtos_externos': ('Uso Organizacional / Negócio', 'IA Integrada a Produtos e Serviços para Clientes', 'Uso Ativo'),
    'ia_principal_frente_negocio': ('Uso Organizacional / Negócio', 'IA como Frente Central de Diferenciação do Negócio', 'Uso Ativo'),
    'ia_nao_prioridade': ('Postura Organizacional', 'IA Não é Prioridade Estratégica no Momento', 'Postura'),
    'falta_compreensao_casos_uso': ('Barreira / Desafio', 'Falta de Compreensão dos Casos de Uso', 'Barreira'),
    'falta_confiabilidade_saidas': ('Barreira / Desafio', 'Falta de Confiabilidade / Alucinações nas Saídas', 'Barreira'),
    'incerteza_regulamentacao': ('Barreira / Desafio', 'Incertezas Regulatórias e Jurídicas', 'Barreira'),
    'preocupacoes_seguranca_privacidade': ('Barreira / Desafio', 'Preocupações com Segurança e Privacidade de Dados', 'Barreira'),
    'roi_nao_comprovado_ia': ('Barreira / Desafio', 'Retorno Financeiro (ROI) Não Comprovado', 'Barreira'),
    'dados_empresa_nao_prontos_ia': ('Barreira / Desafio', 'Dados Despreparados / Falta de Governança', 'Barreira'),
    'falta_expertise_recursos': ('Barreira / Desafio', 'Falta de Expertise Técnica e Mão de Obra', 'Barreira'),
    'alta_direcao_nao_ve_valor': ('Barreira / Desafio', 'Alta Direção Não Vê Valor Imediato', 'Barreira'),
    'preocupacoes_propriedade_intelectual': ('Barreira / Desafio', 'Preocupações com Propriedade Intelectual', 'Barreira')
}
cols_contexto_ia = [c for c in ['respondente_key', 'id', 'ano_pesquisa', 'genero', 'regiao_onde_mora', 'cargo_atual', 'macro_cargo', 'nivel', 'tempo_experiencia_dados', 'faixa_salarial', 'salario_estimado_num', 'modelo_trabalho_resumido', 'setor'] if c in df_spark.columns]
ia_dfs = []
for col, (categoria, label, tipo_registro) in dicionario_ia.items():
    if col in df_spark.columns:
        sub = df_spark.filter(F.col(col).isin("1", "1.0", "true", "True", "SIM", "Sim", "sim", "x", "X")) \
                      .select(cols_contexto_ia) \
                      .withColumn("indicador_ia_id", F.lit(col)) \
                      .withColumn("categoria_ia", F.lit(categoria)) \
                      .withColumn("indicador_ia_label", F.lit(label)) \
                      .withColumn("tipo_registro", F.lit(tipo_registro)) \
                      .withColumn("valor", F.lit(1))
        ia_dfs.append(sub)

spark_spec_ia = ia_dfs[0]
for idf in ia_dfs[1:]:
    spark_spec_ia = spark_spec_ia.unionByName(idf)
spark_spec_ia = spark_spec_ia.dropDuplicates(['respondente_key', 'indicador_ia_id'])
spark_spec_ia.toPandas().to_parquet(SAIDA_SPECS / 'spec_adocao_ia.parquet', index=False)
print(f"  [OK] 3/7 spec_adocao_ia: {spark_spec_ia.count():,} linhas")

# 4. spec_diversidade_carreira
spark_spec_div = df_spark.filter(F.col("genero").isNotNull()) \
                         .groupBy("ano_pesquisa", "macro_cargo", "nivel", "genero", "cor_raca_etnia") \
                         .agg(
                             F.countDistinct("respondente_key").alias("respondentes"),
                             F.round(F.avg("salario_estimado_num"), 2).alias("salario_medio"),
                             F.expr("percentile_approx(salario_estimado_num, 0.5)").alias("salario_mediano"),
                             F.round(100.0 * F.avg(F.when(F.col("modelo_trabalho_resumido") == "Remoto", 1.0).otherwise(0.0)), 1).alias("remoto_pct")
                         )
spark_spec_div.toPandas().to_parquet(SAIDA_SPECS / 'spec_diversidade_carreira.parquet', index=False)
print(f"  [OK] 4/7 spec_diversidade_carreira: {spark_spec_div.count():,} linhas")

# 5. spec_segmentacao_negocio
adotantes_ia_df = spark_spec_ia.filter(F.col("tipo_registro") == "Uso Ativo").select("respondente_key").distinct().withColumn("usa_ia_ativo", F.lit(1.0))
df_resp_ia = df_spark.join(adotantes_ia_df, on="respondente_key", how="left").fillna({"usa_ia_ativo": 0.0}).withColumn("is_mulher", F.when(F.col("genero") == "Feminino", 1.0).otherwise(0.0))
spark_spec_seg = df_resp_ia.groupBy("ano_pesquisa", "macro_cargo", "nivel", "regiao_onde_mora", "modelo_trabalho_resumido") \
                           .agg(
                               F.countDistinct("respondente_key").alias("total_respondentes"),
                               F.round(F.avg("salario_estimado_num"), 2).alias("salario_medio_estimado"),
                               F.expr("percentile_approx(salario_estimado_num, 0.5)").alias("salario_mediano_estimado"),
                               F.round(100.0 * F.avg("usa_ia_ativo"), 1).alias("pct_adotantes_ia"),
                               F.round(100.0 * F.avg("is_mulher"), 1).alias("pct_mulheres")
                           )
spark_spec_seg.toPandas().to_parquet(SAIDA_SPECS / 'spec_segmentacao_negocio.parquet', index=False)
print(f"  [OK] 5/7 spec_segmentacao_negocio: {spark_spec_seg.count():,} linhas")

# 6. spec_dinamica_trabalho_satisfacao
spark_spec_trab = df_spark.withColumn(
    "status_alinhamento",
    F.when(F.col("modelo_trabalho_resumido") == "Não informado", "Não informado")
     .when(F.col("modelo_ideal_resumido") == "Não informado", "Não informado")
     .when(F.col("modelo_trabalho_resumido") == F.col("modelo_ideal_resumido"), "Alinhado (Modelo Satisfeito)")
     .when(F.col("modelo_trabalho_resumido").isin("Presencial", "Híbrido Fixo") & F.col("modelo_ideal_resumido").isin("Remoto", "Híbrido Flexível"), "Deseja Mais Flexibilidade / Remoto")
     .when((F.col("modelo_trabalho_resumido") == "Remoto") & F.col("modelo_ideal_resumido").isin("Presencial", "Híbrido Flexível", "Híbrido Fixo"), "Deseja Mais Presencial / Escritório")
     .otherwise("Outro Descompasso")
).select(
    'respondente_key', 'id', 'ano_pesquisa', 'macro_cargo', 'nivel', 'regiao_onde_mora',
    'setor', 'modelo_trabalho_resumido', 'modelo_ideal_resumido', 'status_alinhamento',
    'oportunidade_buscada', 'salario_estimado_num', 'tempo_experiencia_dados'
)
spark_spec_trab.toPandas().to_parquet(SAIDA_SPECS / 'spec_dinamica_trabalho_satisfacao.parquet', index=False)
print(f"  [OK] 6/7 spec_dinamica_trabalho_satisfacao: {spark_spec_trab.count():,} linhas")

# 7. spec_estrutura_times_empresa
papeis_empresa = {
    'analytics_engineer': 'Analytics Engineer', 'engenheiro_de_dados': 'Engenheiro de Dados', 'analisa_de_dados': 'Analista de Dados', 'cientista_de_dados': 'Cientista de Dados', 'database_administrator': 'DBA / Administrador de Banco', 'analista_de_business': 'Analista de Negócios (BI/BA)', 'arquiteto_de_dados': 'Arquiteto de Dados', 'product_manager': 'Data Product Manager', 'business_analyst': 'Business Analyst', 'ml_engineer': 'Machine Learning Engineer'
}
time_dfs = []
cols_contexto_time = ['respondente_key', 'id', 'ano_pesquisa', 'setor', 'regiao_onde_mora', 'macro_cargo']
for col, label in papeis_empresa.items():
    if col in df_spark.columns:
        sub = df_spark.filter(F.col(col).isin("1", "1.0", "true", "True", "SIM", "Sim", "sim", "x", "X")) \
                      .select(cols_contexto_time) \
                      .withColumn("papel_na_empresa", F.lit(label)) \
                      .withColumn("papel_id", F.lit(col)) \
                      .withColumn("presente_na_empresa", F.lit(True))
        time_dfs.append(sub)

if len(time_dfs) > 0:
    spark_spec_times = time_dfs[0]
    for tdf in time_dfs[1:]:
        spark_spec_times = spark_spec_times.unionByName(tdf)
    spark_spec_times = spark_spec_times.dropDuplicates(['respondente_key', 'papel_na_empresa'])
    spark_spec_times.toPandas().to_parquet(SAIDA_SPECS / 'spec_estrutura_times_empresa.parquet', index=False)
    print(f"  [OK] 7/7 spec_estrutura_times_empresa: {spark_spec_times.count():,} linhas")

spark.stop()
print("\n[OK] Todas as 7 SPECS foram geradas e persistidas com sucesso em:", SAIDA_SPECS)


Gerando as 7 SPECS analiticas...
  [OK] 1/7 spec_respondentes: 14,005 linhas
  [OK] 2/7 spec_adocao_tecnologias: 65,336 linhas
  [OK] 3/7 spec_adocao_ia: 30,355 linhas
  [OK] 4/7 spec_diversidade_carreira: 681 linhas
  [OK] 5/7 spec_segmentacao_negocio: 1,224 linhas
  [OK] 6/7 spec_dinamica_trabalho_satisfacao: 14,005 linhas
  [OK] 7/7 spec_estrutura_times_empresa: 10,623 linhas

[OK] Todas as 7 SPECS foram geradas e persistidas com sucesso em: d:\d\pos\Fase 3\projeto\fase_3_data_analytics\dados\bases_analiticas
